# Wind calibration tutorial (local run)

This notebook shows a practical end-to-end workflow for running wind calibration locally with `WindCalibrator` and checking the generated output files.

## 1) Environment and setup expectations

Before running the notebook:

- Use Python 3.12.
- Install project dependencies (including notebook extras): `uv sync --group notebook`
- Ensure NetCDF support is available (`xarray` + `netCDF4`) because ERA5 loading depends on it.

> If `uv` is not available in your environment, install dependencies with your standard Python package workflow.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from weather.calibration.wind.wind_calibrator import WindCalibrator

## 2) Required input folder structure and files

`WindCalibrator` expects a base `data_path` with these subfolders:

- `plant/plant_data.csv`
- `generation/generation_data.parquet`
- `era5/*.nc` (one or more ERA5 NetCDF files)

In [ ]:
DATA_PATH = Path('data')
OUTPUT_PATH = Path('outputs/wind_calibration')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print('Data path:', DATA_PATH.resolve())
print('Output path:', OUTPUT_PATH.resolve())

In [ ]:
required_files = [
    DATA_PATH / 'plant' / 'plant_data.csv',
    DATA_PATH / 'generation' / 'generation_data.parquet',
]
required_dirs = [DATA_PATH / 'plant', DATA_PATH / 'generation', DATA_PATH / 'era5']

for d in required_dirs:
    print(f'{d}:', 'OK' if d.exists() else 'MISSING')

for f in required_files:
    print(f'{f}:', 'OK' if f.exists() else 'MISSING')

era5_files = sorted((DATA_PATH / 'era5').glob('*.nc'))
print('ERA5 files found:', len(era5_files))
for path in era5_files[:10]:
    print(' -', path.name)
if len(era5_files) > 10:
    print(f' ... and {len(era5_files) - 10} more')

### Optional pre-check: verify ERA5 NetCDF can be opened

If this fails, calibration will fail during ERA5 loading.

In [ ]:
if era5_files:
    test_file = era5_files[0]
    print('Testing ERA5 file:', test_file)
    try:
        ds = xr.open_dataset(test_file, engine='netcdf4')
        print('Opened successfully. Variables:', list(ds.data_vars)[:10])
        ds.close()
    except Exception as exc:
        print('Failed to open ERA5 file with netcdf4:', exc)
else:
    print('No ERA5 files found. Add .nc files under data/era5/.')

## 3) Import and configure `WindCalibrator`

The key arguments are:

- `data_path`: base folder containing `plant/`, `generation/`, `era5/`
- `output_path`: where calibration outputs are written
- `visual_output`: save power curve PNG plots when `True`
- `stream_npy_output`: also save `Wind Streams.npy` when `True`

In [ ]:
wc = WindCalibrator(
    data_path=str(DATA_PATH),
    output_path=OUTPUT_PATH,
    visual_output=True,
    stream_npy_output=True,
)
wc

## 4) Run calibration

This executes the full workflow and writes outputs to `output_path`.

In [ ]:
wc.calibrate()

## 5) Inspect generated outputs

Core output files:

- `Calibration Summary.csv`
- `Weibull Params.csv`
- `Wind Speeds.csv`
- `Wind Streams.parquet`
- optional: PNG plots and `Wind Streams.npy`

In [ ]:
sorted([p.name for p in OUTPUT_PATH.glob('*')])

In [ ]:
summary = pd.read_csv(OUTPUT_PATH / 'Calibration Summary.csv')
summary.head()

In [ ]:
weibull = pd.read_csv(OUTPUT_PATH / 'Weibull Params.csv')
weibull.head()

In [ ]:
wind_speeds = pd.read_csv(OUTPUT_PATH / 'Wind Speeds.csv')
wind_speeds.head()

In [ ]:
wind_streams = pd.read_parquet(OUTPUT_PATH / 'Wind Streams.parquet')
wind_streams.head()

In [ ]:
png_outputs = sorted(OUTPUT_PATH.glob('*.png'))
npy_outputs = sorted(OUTPUT_PATH.glob('*.npy'))
print('PNG outputs:', [p.name for p in png_outputs[:10]])
if len(png_outputs) > 10:
    print(f'... and {len(png_outputs) - 10} more PNG files')
print('NPY outputs:', [p.name for p in npy_outputs])

## 6) Basic troubleshooting

### ERA5 / NetCDF loading issues

If you see warnings like "Failed to load ... xarray IO backends ...":

1. Confirm dependencies are installed (including notebook extras):
   - `uv sync --group notebook`
2. Verify `netCDF4` is importable and print versions:
3. Try opening one ERA5 file with `engine='netcdf4'` (cell above).
4. Confirm `.nc` files are in `data/era5/` and not corrupted.

If no files load successfully, `WindCalibrator` raises: `ValueError: Failed to load any NetCDF files successfully`.

In [ ]:
import importlib

for pkg in ['xarray', 'netCDF4', 'h5netcdf']:
    spec = importlib.util.find_spec(pkg)
    print(f'{pkg}:', 'installed' if spec else 'missing')

### Other quick checks

- Missing plant file: `data/plant/plant_data.csv`
- Missing generation file: `data/generation/generation_data.parquet`
- Missing ERA5 directory/files: `data/era5/*.nc`
- Permission issues writing output: ensure `output_path` is writable